# Waste Segregation - Wet and Dry Waste Detection

## Project Info
- **Repo**: https://github.com/raushan95a/Waste-segregation
- **Dataset Location**: Automatically detected (works locally or in Google Drive for Colab)
- **Run cells top to bottom**

This notebook is compatible with both **Google Colab** and **local execution**.

## Step 1: Environment Setup

This section automatically detects if you're running in Google Colab or locally and sets up the appropriate paths.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Google Drive folder ID for the dataset
GOOGLE_DRIVE_FOLDER_ID = "1VJE5qj9DjZ9rZy6JT_-AIWVTwgbnRLDl"

# Detect environment and set up paths
IS_COLAB = False
DATASET_ROOT = None

try:
    from google.colab import drive
    IS_COLAB = True
    print("✓ Google Colab detected - mounting Google Drive...")
    drive.mount('/content/drive')
    
    # First try the default path
    default_path = '/content/drive/My Drive/Waste_Dataset'
    if os.path.exists(default_path):
        DATASET_ROOT = default_path
        print(f"✓ Found dataset at: {DATASET_ROOT}")
    else:
        # Try to download from the provided Google Drive folder
        print(f"Downloading dataset from Google Drive (ID: {GOOGLE_DRIVE_FOLDER_ID})...")
        try:
            import subprocess
            subprocess.run(["pip", "install", "gdown", "-q"], check=True)
            import gdown
            
            # Download the folder
            os.makedirs('/content/drive/My Drive/Waste_Dataset', exist_ok=True)
            gdown.download_folder(
                f"https://drive.google.com/drive/folders/{GOOGLE_DRIVE_FOLDER_ID}?usp=drive_link",
                output='/content/drive/My Drive/Waste_Dataset',
                quiet=False,
                use_cookies=False
            )
            DATASET_ROOT = '/content/drive/My Drive/Waste_Dataset'
            print(f"✓ Dataset downloaded to: {DATASET_ROOT}")
        except Exception as e:
            print(f"✗ Download failed: {e}")
            print("Trying alternative download method...")
            DATASET_ROOT = '/content/drive/My Drive/Waste_Dataset'
    
    os.chdir('/content/drive/My Drive')
    
except ImportError:
    print("✓ Local environment detected")
    # For local execution, construct path relative to current location
    DATASET_ROOT = os.path.abspath('Waste_Dataset')
    
# Verify dataset exists
if not os.path.exists(DATASET_ROOT):
    print(f"\n⚠ Dataset not found at: {DATASET_ROOT}")
    print(f"  Colab users: Make sure the Google Drive folder is accessible")
    print(f"  Local users: Ensure Waste_Dataset folder exists at: {DATASET_ROOT}")
else:
    print(f"✓ Dataset found at: {DATASET_ROOT}")
    # List contents
    if os.path.exists(os.path.join(DATASET_ROOT, 'Images_merged')):
        images = len([f for f in os.listdir(os.path.join(DATASET_ROOT, 'Images_merged')) if f.endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  Images: {images} files")
    if os.path.exists(os.path.join(DATASET_ROOT, 'Annotations_merged')):
        annos = len([f for f in os.listdir(os.path.join(DATASET_ROOT, 'Annotations_merged')) if f.endswith('.xml')])
        print(f"  Annotations: {annos} files")

print(f"\nEnvironment: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset root: {DATASET_ROOT}")

Installation instructions:

Run these commands:

git clone https://github.com/Tessellate-Imaging/Monk_Object_Detection.git

cd Monk_Object_Detection/5_pytorch_retinanet/installation

Select the right requirements file and run

cat requirements_cuda.txt | xargs -n 1 -L 1 pip install

In [ ]:
! git clone https://github.com/Tessellate-Imaging/Monk_Object_Detection.git

In [ ]:
# For colab use the command below
! cd Monk_Object_Detection/5_pytorch_retinanet/installation && cat requirements_colab.txt | xargs -n 1 -L 1 pip install

# For Local systems and cloud select the other file
#! cd Monk_Object_Detection/5_pytorch_retinanet/installation && cat requirements.txt | xargs -n 1 -L 1 pip install

In [ ]:
## Step 2: Install Required Dependencies

import subprocess
import sys

def install_package(package):
    """Install a package with error handling"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        return True
    except Exception as e:
        print(f"Warning: Could not install {package}: {e}")
        return False

print("Installing dependencies...")
packages = ['xmltodict', 'pycocotools', 'tqdm', 'opencv-python', 'numpy', 'pandas']

for package in packages:
    if install_package(package):
        print(f"✓ {package}")

print("Dependencies installation complete!")

## Step 3: Import Libraries and Set Paths

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import xmltodict
import json
from tqdm.notebook import tqdm
from pathlib import Path

# Set working directory based on environment
if not IS_COLAB:
    # For local execution, change to the workspace directory
    workspace_dir = Path(__file__).parent if hasattr(__file__, '__fspath__') else Path.cwd()
    if 'Waste-segregation' in str(workspace_dir):
        os.chdir(workspace_dir)
    else:
        os.chdir('Waste-segregation') if os.path.exists('Waste-segregation') else None

print(f"Working directory: {os.getcwd()}")

# Define dataset paths
root_dir = DATASET_ROOT
img_dir = os.path.join(DATASET_ROOT, 'Images_merged')
anno_dir = os.path.join(DATASET_ROOT, 'Annotations_merged')

# Verify paths exist
for path, name in [(root_dir, 'Dataset'), (img_dir, 'Images'), (anno_dir, 'Annotations')]:
    if os.path.exists(path):
        print(f"✓ {name} directory found: {path}")
    else:
        print(f"✗ {name} directory NOT found: {path}")

In [ ]:
# List annotation files
try:
    files = os.listdir(anno_dir)
    print(f"Found {len(files)} annotation files")
    print(f"Sample files: {files[:5]}")  # Show first 5 files
except FileNotFoundError:
    print(f"Error: Annotations directory not found at {anno_dir}")
    files = []
except Exception as e:
    print(f"Error listing files: {e}")
    files = []

## Step 4: Convert Annotations from Pascal VOC to COCO Format

All annotation files are in Pascal VOC XML format. We'll convert them to COCO format for model training.

### Convert VOC annotations to intermediate format

First, we parse XML files and create a CSV with bounding box annotations.

In [ ]:
import subprocess
import sys

# Ensure xmltodict is installed
try:
    import xmltodict
    print("✓ xmltodict already installed")
except ImportError:
    print("Installing xmltodict...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xmltodict", "-q"])
    import xmltodict
    print("✓ xmltodict installed")

NOTE: When running the kernel, if it says, module not found for any library, just use !pip install that library name, or apt install.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import xmltodict
import json
from tqdm.notebook import tqdm
from pathlib import Path

print("✓ All libraries imported successfully")

The path of the directories, will be the according to the name of the dataset folder in the drive.

In [ ]:
# Set paths for annotation conversion
root_dir = DATASET_ROOT
img_dir_rel = "Images_merged"
anno_dir_rel = "Annotations_merged"

print(f"Dataset root: {root_dir}")
print(f"Images directory: {os.path.join(root_dir, img_dir_rel)}")
print(f"Annotations directory: {os.path.join(root_dir, anno_dir_rel)}")

In [ ]:
# List annotation files
anno_path = os.path.join(root_dir, anno_dir_rel)
try:
    files = os.listdir(anno_path)
    files = [f for f in files if f.endswith('.xml')]  # Only XML files
    print(f"Found {len(files)} annotation files")
    if files:
        print(f"Sample files: {files[:3]}")
except Exception as e:
    print(f"Error: {e}")
    print(f"Path checked: {anno_path}")
    files = []

In [ ]:
## Convert VOC XML to intermediate format

combined = []
anno_path = os.path.join(root_dir, anno_dir_rel)
errors = []

for i in tqdm(range(len(files)), desc="Processing annotations"):
    try:
        xml_file = os.path.join(anno_path, files[i])
        
        with open(xml_file, 'r', encoding='utf-8') as f:
            my_xml = f.read()
        
        anno = dict(dict(xmltodict.parse(my_xml))["annotation"])
        fname = anno.get("filename", "")
        
        if not fname:
            errors.append(f"{files[i]}: No filename found")
            continue
            
        label_str = ""
        
        # Handle single or multiple objects
        objects = anno.get("object", [])
        if not isinstance(objects, list):
            objects = [objects]
        
        for j, obj in enumerate(objects):
            try:
                label = obj.get("name", "unknown")
                bbox = obj.get("bndbox", {})
                
                x1 = str(bbox.get("xmin", 0))
                y1 = str(bbox.get("ymin", 0))
                x2 = str(bbox.get("xmax", 0))
                y2 = str(bbox.get("ymax", 0))
                
                bbox_str = f"{x1} {y1} {x2} {y2} {label}"
                
                if j == len(objects) - 1:
                    label_str += bbox_str
                else:
                    label_str += bbox_str + " "
            except Exception as e:
                errors.append(f"{files[i]}, object {j}: {e}")
                continue
        
        if label_str:  # Only add if we found bounding boxes
            combined.append([fname, label_str])
            
    except Exception as e:
        errors.append(f"{files[i]}: {e}")
        continue

print(f"✓ Processed {len(combined)} annotations successfully")
if errors:
    print(f"⚠ Encountered {len(errors)} errors:")
    for err in errors[:5]:  # Show first 5 errors
        print(f"  - {err}")

In [ ]:
# Display sample of processed data
print(f"Total annotations: {len(combined)}")
print("\nSample annotations:")
for i in range(min(3, len(combined))):
    print(f"  {combined[i][0]}: {combined[i][1][:100]}...")  # Show first 100 chars

In [ ]:
# Save to CSV
try:
    df = pd.DataFrame(combined, columns=['ID', 'Label'])
    csv_path = os.path.join(root_dir, "train_labels.csv")
    df.to_csv(csv_path, index=False)
    print(f"✓ Saved {len(df)} annotations to: {csv_path}")
    print(f"  DataFrame shape: {df.shape}")
except Exception as e:
    print(f"✗ Error saving CSV: {e}")
    print(f"  Target path: {csv_path}")

In [ ]:
print("\n" + "="*60)
print("CONVERSION TO COCO FORMAT")
print("="*60)

### Convert to COCO JSON format

Convert the intermediate format to COCO format for training.

Import necessary libraries for conversion

In [ ]:
import os
import numpy as np 
import cv2
import json
import pandas as pd
from pathlib import Path

print("✓ All conversion libraries imported")

In [ ]:
# Set paths for COCO conversion
root = DATASET_ROOT
img_dir = "Images_merged"
anno_file = "train_labels.csv"

print(f"Root: {root}")
print(f"Images dir: {img_dir}")
print(f"Annotation file: {anno_file}")

In [ ]:
# Define full paths
dataset_path = root
images_folder = os.path.join(root, img_dir)
annotations_path = os.path.join(root, "annotations")

print(f"Dataset path: {dataset_path}")
print(f"Images folder: {images_folder}")
print(f"Annotations path: {annotations_path}")

In [ ]:
# Create annotations directory if it doesn't exist
try:
    os.makedirs(annotations_path, exist_ok=True)
    print(f"✓ Annotations directory ready: {annotations_path}")
except Exception as e:
    print(f"✗ Error creating directory: {e}")

# Verify input files exist
input_images_folder = images_folder
input_annotations_path = os.path.join(root, anno_file)

print(f"\nInput images folder: {input_images_folder} (exists: {os.path.exists(input_images_folder)})")
print(f"Input CSV file: {input_annotations_path} (exists: {os.path.exists(input_annotations_path)})")

In [ ]:
# Define output paths
output_dataset_path = root
output_image_folder = input_images_folder
output_annotation_folder = annotations_path

tmp = img_dir.replace("/", "")
output_annotation_file = os.path.join(output_annotation_folder, f"instances_{tmp}.json")
output_classes_file = os.path.join(output_annotation_folder, "classes.txt")

print(f"Output annotation JSON: {output_annotation_file}")
print(f"Output classes file: {output_classes_file}")

In [ ]:
# Ensure output directory exists
os.makedirs(output_annotation_folder, exist_ok=True)
print(f"✓ Output directory ready")

In [ ]:
# Read the CSV with annotations
try:
    df = pd.read_csv(input_annotations_path)
    columns = df.columns
    print(f"✓ Loaded CSV with {len(df)} rows")
    print(f"  Columns: {list(columns)}")
except FileNotFoundError:
    print(f"✗ CSV file not found: {input_annotations_path}")
    df = None
except Exception as e:
    print(f"✗ Error reading CSV: {e}")
    df = None

In [ ]:
# Define delimiter for bounding box parsing
delimiter = " "
print(f"Using delimiter: '{delimiter}'")

In [ ]:
# Extract unique categories from annotations
list_dict = []
anno = []
errors = []

if df is not None:
    for i in range(len(df)):
        try:
            img_name = df[columns[0]][i]
            labels = str(df[columns[1]][i])
            tmp = labels.split(delimiter)
            
            # Extract category labels (every 5th element starting from index 4)
            for j in range(len(tmp)//5):
                label = tmp[j*5+4]
                if label not in anno:
                    anno.append(label)
        except Exception as e:
            errors.append(f"Row {i}: {e}")

    # Sort categories
    anno = sorted(anno)
    
    # Create category dictionaries for COCO format
    for i in range(len(anno)):
        tmp = {}
        tmp["supercategory"] = "waste"
        tmp["id"] = i
        tmp["name"] = anno[i]
        list_dict.append(tmp)

    # Save classes file
    try:
        with open(output_classes_file, 'w') as anno_f:
            for label in anno:
                anno_f.write(label + "\n")
        print(f"✓ Found {len(anno)} unique classes: {anno}")
        print(f"✓ Saved classes to: {output_classes_file}")
    except Exception as e:
        print(f"✗ Error saving classes file: {e}")

    if errors:
        print(f"\n⚠ Encountered {len(errors)} errors during parsing")
        for err in errors[:3]:
            print(f"  - {err}")
else:
    print("✗ Cannot process - CSV file not loaded")

In [ ]:
# Create COCO format JSON
coco_data = {}
coco_data["type"] = "instances"
coco_data["images"] = []
coco_data["annotations"] = []
coco_data["categories"] = list_dict

image_id = 0
annotation_id = 0
processed_images = 0
failed_images = 0
skipped_images = 0

if df is not None:
    for i in tqdm(range(len(df)), desc="Creating COCO dataset"):
        try:
            img_name = df[columns[0]][i]
            labels = str(df[columns[1]][i])
            tmp = labels.split(delimiter)
            image_in_path = os.path.join(input_images_folder, img_name)
            
            # Check if image file exists
            if not os.path.exists(image_in_path):
                failed_images += 1
                continue
            
            # Read image to get dimensions
            img = cv2.imread(image_in_path, 1)
            if img is None:
                failed_images += 1
                continue
                
            h, w, c = img.shape
            
            # Add image info
            images_tmp = {}
            images_tmp["file_name"] = img_name
            images_tmp["height"] = h
            images_tmp["width"] = w
            images_tmp["id"] = image_id
            coco_data["images"].append(images_tmp)
            
            # Add annotations
            for j in range(len(tmp)//5):
                try:
                    x1 = int(float(tmp[j*5+0]))
                    y1 = int(float(tmp[j*5+1]))
                    x2 = int(float(tmp[j*5+2]))
                    y2 = int(float(tmp[j*5+3]))
                    label = tmp[j*5+4]
                    
                    # Validate bounding box
                    if x2 <= x1 or y2 <= y1:
                        continue
                    
                    annotations_tmp = {}
                    annotations_tmp["id"] = annotation_id
                    annotation_id += 1
                    annotations_tmp["image_id"] = image_id
                    annotations_tmp["segmentation"] = []
                    annotations_tmp["ignore"] = 0
                    annotations_tmp["area"] = (x2-x1)*(y2-y1)
                    annotations_tmp["iscrowd"] = 0
                    annotations_tmp["bbox"] = [x1, y1, x2-x1, y2-y1]
                    annotations_tmp["category_id"] = anno.index(label) if label in anno else 0
                    
                    coco_data["annotations"].append(annotations_tmp)
                except (ValueError, IndexError) as e:
                    continue
            
            image_id += 1
            processed_images += 1
            
        except Exception as e:
            failed_images += 1
            continue

    # Save COCO JSON file
    try:
        with open(output_annotation_file, 'w') as outfile:
            json_str = json.dumps(coco_data, indent=4)
            outfile.write(json_str)
        
        print(f"✓ COCO dataset created successfully!")
        print(f"  Images processed: {processed_images}")
        print(f"  Total annotations: {len(coco_data['annotations'])}")
        print(f"  Total images in COCO: {len(coco_data['images'])}")
        print(f"  Failed/missing images: {failed_images}")
        print(f"  Saved to: {output_annotation_file}")
        
    except Exception as e:
        print(f"✗ Error saving COCO JSON: {e}")
else:
    print("✗ Cannot create COCO dataset - CSV not loaded")

In [ ]:
## Step 5: Train Object Detection Model

We'll use the Monk Object Detection library with PyTorch RetinaNet for training.

### Setup and Training

Clone the Monk Object Detection repository and set up the training pipeline.

In [ ]:
## Step 6: Clone and Setup Monk Object Detection

import os
import subprocess
import sys

# Check if Monk_Object_Detection already exists
monk_path = "Monk_Object_Detection"
if not os.path.exists(monk_path):
    print("Cloning Monk Object Detection repository...")
    try:
        subprocess.run(["git", "clone", 
                       "https://github.com/Tessellate-Imaging/Monk_Object_Detection.git"],
                      check=True)
        print("✓ Repository cloned successfully")
    except Exception as e:
        print(f"✗ Error cloning repository: {e}")
        print("  Using pre-existing installation or continuing without Monk...")
else:
    print(f"✓ Monk repository already exists at: {monk_path}")

# Add to path
monk_lib_path = os.path.join(monk_path, "5_pytorch_retinanet", "lib")
if monk_lib_path not in sys.path:
    sys.path.append(monk_lib_path)
    print(f"✓ Added to sys.path: {monk_lib_path}")

In [ ]:
## Initialize Training Module

try:
    from train_detector import Detector
    gtf = Detector()
    print("✓ Training module initialized successfully")
except ImportError as e:
    print(f"✗ Error importing Detector: {e}")
    print("  Make sure Monk Object Detection is properly installed")
    gtf = None
except Exception as e:
    print(f"✗ Unexpected error: {e}")
    gtf = None

In [ ]:
### Configure Training Dataset

In [ ]:
# Configure dataset paths for training
if gtf is not None:
    root_dir = "./"
    coco_dir = "Waste_Dataset"
    img_dir = "./"
    set_dir = "Images_merged"
    
    print(f"Training configuration:")
    print(f"  Root dir: {root_dir}")
    print(f"  COCO dir: {coco_dir}")
    print(f"  Image dir: {img_dir}")
    print(f"  Set dir: {set_dir}")

In [ ]:
# Load training dataset
if gtf is not None:
    try:
        print("Loading training dataset...")
        gtf.Train_Dataset(root_dir, coco_dir, img_dir, set_dir, batch_size=2, use_gpu=True)
        print("✓ Dataset loaded successfully")
    except Exception as e:
        print(f"✗ Error loading dataset: {e}")
        print("  Check that COCO annotation files are properly formatted")
else:
    print("✗ Training module not initialized")

In [ ]:
# Display detected classes
if gtf is not None:
    try:
        classes = gtf.system_dict["local"]["dataset_train"].classes
        print(f"✓ Detected {len(classes)} waste classes:")
        for i, cls in enumerate(classes):
            print(f"  {i}: {cls}")
    except Exception as e:
        print(f"Note: Could not retrieve classes: {e}")

### Select Model Architecture

Using ResNet50 backbone for object detection with RetinaNet

In [ ]:
# Set model architecture
if gtf is not None:
    try:
        print("Setting model: ResNet50...")
        gtf.Model(model_name="resnet50")
        print("✓ Model configured: ResNet50")
    except Exception as e:
        print(f"✗ Error configuring model: {e}")

In [ ]:
# Set training hyperparameters
if gtf is not None:
    try:
        print("Setting hyperparameters...")
        gtf.Set_Hyperparams(lr=0.0001, print_interval=20)
        print("✓ Hyperparameters set:")
        print("  Learning rate: 0.0001")
        print("  Print interval: 20 batches")
    except Exception as e:
        print(f"✗ Error setting hyperparameters: {e}")

### Train the Model

Training for 8 epochs. This may take some time depending on dataset size and hardware.

In [ ]:
# Train the model
if gtf is not None:
    try:
        print("Starting training...")
        print("This may take a significant amount of time based on dataset size and hardware\n")
        gtf.Train(num_epochs=8, output_model_name="final_model.pt")
        print("\n✓ Training completed successfully!")
        print("✓ Model saved as: final_model.pt")
    except Exception as e:
        print(f"✗ Training error: {e}")
        print("  Check the logs above for details")

In [ ]:
## Step 7: Inference on Test Images

Use the trained model to make predictions on new waste images.

In [ ]:
### Setup Inference Module

import os
import sys

# Add Monk library to path
monk_lib_path = os.path.join("Monk_Object_Detection", "5_pytorch_retinanet", "lib")
if monk_lib_path not in sys.path:
    sys.path.append(monk_lib_path)

print(f"Inference environment ready")

In [ ]:
# Import inference module
try:
    from infer_detector import Infer
    print("✓ Inference module imported successfully")
except ImportError as e:
    print(f"✗ Error importing inference module: {e}")
    print("  Make sure Monk Object Detection inference module is available")

In [ ]:
# Initialize inference object
try:
    gtf_infer = Infer()
    print("✓ Inference object initialized")
except Exception as e:
    print(f"✗ Error initializing inference: {e}")
    gtf_infer = None

In [ ]:
# Load trained model
if gtf_infer is not None:
    model_path = "final_model.pt"
    
    if os.path.exists(model_path):
        try:
            print(f"Loading model: {model_path}...")
            result = gtf_infer.Model(model_path=model_path)
            print(f"✓ Model loaded successfully")
            print(f"  Result: {result}")
        except Exception as e:
            print(f"✗ Error loading model: {e}")
    else:
        print(f"✗ Model file not found: {model_path}")
        print(f"  Please ensure the model has been trained first")
else:
    print("✗ Inference object not initialized")

In [ ]:
print(gtf.Model(model_path="final_model.pt"))

In [ ]:
# Load class labels
classes_file = os.path.join(DATASET_ROOT, "annotations", "classes.txt")

try:
    with open(classes_file, 'r') as f:
        class_list = f.readlines()
    
    # Clean up class names
    class_list = [cls.strip() for cls in class_list]
    
    print(f"✓ Loaded {len(class_list)} classes:")
    for i, cls in enumerate(class_list):
        print(f"  {i}: {cls}")
        
except FileNotFoundError:
    print(f"✗ Classes file not found: {classes_file}")
    print(f"  Please ensure annotations have been converted to COCO format")
    class_list = []
except Exception as e:
    print(f"✗ Error loading classes: {e}")
    class_list = []

In [ ]:
# Verify classes were loaded
if class_list:
    print(f"\n✓ Ready for predictions with {len(class_list)} classes")
else:
    print("\n✗ No classes available for predictions")

In [ ]:
### Test Predictions on Sample Images

Run predictions on sample images from the dataset. You can modify the image paths to test on different images.

#### Sample Image Predictions

Change the `img_path` variable to test on different images in the dataset.

In [ ]:
# Prediction helper function
def predict_and_display(img_filename, vis_threshold=0.4):
    """Run prediction on an image and display results"""
    img_path = os.path.join(DATASET_ROOT, "Images_merged", img_filename)
    
    if not os.path.exists(img_path):
        print(f"✗ Image not found: {img_path}")
        return None
    
    if gtf_infer is None or not class_list:
        print("✗ Model or classes not initialized")
        return None
    
    try:
        print(f"Predicting on: {img_filename}")
        scores, labels, boxes = gtf_infer.Predict(img_path, class_list, vis_threshold=vis_threshold)
        
        # Display results
        print(f"✓ Predictions made:")
        print(f"  Detections: {len(boxes)}")
        for i, (label, score, box) in enumerate(zip(labels, scores, boxes)):
            print(f"    {i+1}. {label}: {score:.2%}")
        
        # Display output image if it exists
        try:
            from IPython.display import Image, display
            if os.path.exists('output.jpg'):
                display(Image(filename='output.jpg'))
        except:
            print("  (Output image could not be displayed)")
            
        return scores, labels, boxes
        
    except Exception as e:
        print(f"✗ Prediction error: {e}")
        return None

# Test on first image
test_images = [f for f in os.listdir(os.path.join(DATASET_ROOT, "Images_merged")) 
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if test_images:
    print(f"Found {len(test_images)} images in dataset")
    print(f"\nRunning prediction on: {test_images[0]}")
    predict_and_display(test_images[0], vis_threshold=0.4)
else:
    print("✗ No images found in dataset")

In [ ]:
predict_and_display("plastic258.jpg", vis_threshold=0.4)

In [ ]:
predict_and_display("plastic249.jpg", vis_threshold=0.4)

In [ ]:
predict_and_display("plastic236.jpg", vis_threshold=0.4)

In [ ]:
predict_and_display("O_41.jpg", vis_threshold=0.4)

In [ ]:
predict_and_display("O_37.jpg", vis_threshold=0.4)

In [ ]:
predict_and_display("image107.jpg", vis_threshold=0.4)